## Keras 3 - MADE
This code is an implementation of ["Masked AutoEncoder for Density Estimation"](https://arxiv.org/abs/1502.03509) by Germain et al., 2015. The core idea is that you can turn an auto-encoder into an autoregressive density model just by appropriately masking the connections in the MLP, ordering the input dimensions in some way and making sure that all outputs only depend on inputs earlier in the list. Like other autoregressive models (char-rnn, pixel cnns, etc), evaluating the likelihood is very cheap (a single forward pass), but sampling is linear in the number of dimensions.

**This notebook is a keras3 implementation of [code by Andrej karpathy](https://github.com/karpathy/pytorch-made/blob/master/made.py)**

In [1]:
import jax
import jax.numpy as jnp
import keras
from keras import ops
import keras.layers as L
import numpy as np
from functools import partial

In [2]:
print(keras.backend.backend())

jax


## Define MADE model

The Made model needs a masked linear layer to be created followed by the model

### Masked Linear layer 

In [135]:
class MaskedLinear(L.Dense):
    def __init__(self,units,
        activation=None,
        use_bias=True,
        kernel_initializer='glorot_uniform',
        bias_initializer='zeros',
        kernel_regularizer=None,
        bias_regularizer=None,
        activity_regularizer=None,
        kernel_constraint=None,
        bias_constraint=None,
        lora_rank=None, **kwargs):
        super().__init__(units=units, activation=activation,
                         use_bias=use_bias,
                         kernel_initializer=kernel_initializer,
                         bias_initializer=bias_initializer,
                         kernel_regularizer=kernel_regularizer,
                         bias_regularizer=bias_regularizer,
                         activity_regularizer=activity_regularizer,
                         kernel_constraint=kernel_constraint,
                         bias_constraint=bias_constraint,
                         lora_rank=lora_rank,**kwargs)
        
    def build(self,input_shape):
        super().build(input_shape)
        ## Transpose the shape compared to the kernel matrix for matmult to be correctly applied
        self.mask = self.add_weight(name=f'{self.name}_Mask',shape=self.kernel.shape,
                                   initializer=keras.initializers.ones(),
                                   trainable=False)
    def set_mask(self,mask):
        self.mask.assign(ops.cast(mask,'uint8'))
        
    def call(self,x):
        masked_kernel = ops.multiply(self.kernel,self.mask)
        x = ops.matmul(x,masked_kernel) 
        if self.bias is not None:
            x = ops.add(x, self.bias)
        if self.activation is not None:
            x = self.activation(x)
        return x

### MADE Model
The direct translation from pytorch would be a subclassed model, but I'm taking a shortcut and creating a functional model instead

In [136]:
class MADE:
    def __init__(self,nin, hidden_sizes, nout, num_masks=1, natural_ordering=False):
        """
        nin: integer; number of inputs
        hidden sizes: a list of integers; number of units in hidden layers
        nout: integer; number of outputs, which usually collectively parameterize some kind of 1D distribution
              note: if nout is e.g. 2x larger than nin (perhaps the mean and std), then the first nin
              will be all the means and the second nin will be stds. i.e. output dimensions depend on the
              same input dimensions in "chunks" and should be carefully decoded downstream appropriately.
              the output of running the tests for this file makes this a bit more clear with examples.
        num_masks: can be used to train ensemble over orderings/connections
        natural_ordering: force natural ordering of dimensions, don't use random permutations
        """
        self.nin = nin
        self.nout = nout
        self.hidden_sizes = hidden_sizes
        assert self.nout % self.nin == 0, "nout must be integer multiple of nin"
        ## Create Model
        self.model = self.create_model()
        # seeds for orders/connectivities of the model ensemble
        self.natural_ordering = natural_ordering
        self.num_masks = num_masks
        self.seed = 0 # for cycling through num_masks orderings

        self.m = {}
        self.masks = self.update_masks() # builds the initial self.m connectivity
        self.apply_masks(self.masks)
        
    def create_model(self):
        model = keras.Sequential()
        model.add(L.Input(shape=(self.nin,)))
        for hd in self.hidden_sizes:
            model.add(MaskedLinear(units=hd,activation="relu"))
        model.add(MaskedLinear(units=self.nout, activation="linear"))
        return model
    
    def update_masks(self):
        # only a single seed, skip for efficiency
        if self.m and self.num_masks==1:
            return

        l_hidden = len(self.hidden_sizes)
        # fetch the next seed and construct a random stream
        rng = np.random.RandomState(self.seed)
        self.seed = (self.seed + 1) % self.num_masks

        # sample the order of the inputs and the connectivity of all neurons
        self.m[-1] = np.arange(self.nin) if self.natural_ordering else rng.permutation(self.nin)

        for l in range(l_hidden):
            self.m[l] = rng.randint(self.m[l-1].min(), self.nin-1, size=self.hidden_sizes[l])
            
        # construct the mask matrices
        masks = [self.m[l-1][:,None] <= self.m[l][None,:] for l in range(l_hidden)]
        masks.append(self.m[l_hidden-1][:,None] < self.m[-1][None,:])

        # handle the case where nout = nin * k, for integer k > 1
        if self.nout > self.nin:
            k = int(self.nout / self.nin)
            # replicate the mask across the other outputs
            masks[-1] = np.concatenate([masks[-1]]*k, axis=1)
        return masks
        
    def apply_masks(self,masks):
        # set the masks in all MaskedLinear layers
        layers = [l for l in self.model.layers if isinstance(l, MaskedLinear)]
        for l,m in zip(layers, masks):
            l.set_mask(m)
        
    def summary(self):
        self.model.summary()

### Testing Autoregressive Property 

We check if the masking is correctly creating independence between variables by inspecting the gradients of the outputs w.r.t the inputs. 

**JAX housekeeping**

JAX needs definitions for stateless versions of the forward pass to be usable. We define the `JAXTrainer` helper class to wrap the model inference

In [139]:
class JAXTrainer:
    def __init__(self,model):
        def compute_loss_and_updates(trainable_variables, non_trainable_variables, x, k):
            y_pred, non_trainable_variables = model.stateless_call(
                trainable_variables, non_trainable_variables, x, training=True
            )
            loss = y_pred[0, k]
            return loss, non_trainable_variables
            
        ## Get derivates w.rt. the weights as well as the input.
        grad_fn = jax.value_and_grad(compute_loss_and_updates,argnums=(0,2), has_aux=True)
        
        @jax.jit
        def step_fn(state, x, k):
            trainable_variables, non_trainable_variables = state
            (loss, non_trainable_variables), (weight_grads,input_grads) = grad_fn(
                trainable_variables, non_trainable_variables, x, k
            )
            return loss, (
                trainable_variables,
                non_trainable_variables),weight_grads,input_grads
        # 4. Save the compiled step function to the instance
        self._compiled_train_step = step_fn
        
    def train_step(self, state, x, k):
        return self._compiled_train_step(state, x, k)

In [140]:
nin, hiddens, nout, natural_ordering = D, [200], D, False

In [141]:
# run a quick and dirty test for the autoregressive property
D = 10
rng = np.random.RandomState(14)
x = jnp.array((rng.rand(1, D) > 0.5).astype(np.float32))

configs = [
    (D, [], D, False),                 # test various hidden sizes
    (D, [200], D, False),
    (D, [200, 220], D, False),
    (D, [200, 220, 230], D, False),
    (D, [200, 220], D, True),          # natural ordering test
    (D, [200, 220], 2*D, True),       # test nout > nin
    (D, [200, 220], 3*D, False),       # test nout > nin
]

for nin, hiddens, nout, natural_ordering in configs:
    
    print("checking nin %d, hiddens %s, nout %d, natural %s" % 
         (nin, hiddens, nout, natural_ordering))
    made = MADE(nin, hiddens, nout, natural_ordering=natural_ordering)
    trainable_variables = [v.value for v in made.model.trainable_variables]
    non_trainable_variables = [v.value for v in made.model.non_trainable_variables]
    state = trainable_variables, non_trainable_variables
    jax_trainer = JAXTrainer(made.model)
    
    # run backpropagation for each dimension to compute what other
    # dimensions it depends on.
    res = []
    for k in range(nout):
        loss, state ,  weight_grads, input_grads = jax_trainer.train_step(state,x, k)
        
        depends = (input_grads != 0).astype(np.uint8)
        depends_ix = jnp.where(depends)[1].tolist()
        isok = k % nin not in depends_ix
        
        res.append((len(depends_ix), k, depends_ix, isok))
    
    # pretty print the dependencies
    res.sort()
    for nl, k, ix, isok in res:
        print("output %2d depends on inputs: %30s : %s" % (k, ix, "OK" if isok else "NOTOK"))

checking nin 10, hiddens [], nout 10, natural False
output  8 depends on inputs:                             [] : OK
output  4 depends on inputs:                            [8] : OK
output  0 depends on inputs:                         [4, 8] : OK
output  7 depends on inputs:                      [0, 4, 8] : OK
output  2 depends on inputs:                   [0, 4, 7, 8] : OK
output  9 depends on inputs:                [0, 2, 4, 7, 8] : OK
output  5 depends on inputs:             [0, 2, 4, 7, 8, 9] : OK
output  6 depends on inputs:          [0, 2, 4, 5, 7, 8, 9] : OK
output  1 depends on inputs:       [0, 2, 4, 5, 6, 7, 8, 9] : OK
output  3 depends on inputs:    [0, 1, 2, 4, 5, 6, 7, 8, 9] : OK
checking nin 10, hiddens [200], nout 10, natural False
output  8 depends on inputs:                             [] : OK
output  4 depends on inputs:                            [8] : OK
output  0 depends on inputs:                         [4, 8] : OK
output  7 depends on inputs:                    

### Training the model